In [4]:
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# 1. Load
df = pd.read_csv("trainsvc.csv")
df = df.drop(columns="Loan_ID") # identifier, not a feature
df["Loan_Status"] = df["Loan_Status"].map({"Y": 1, "N": 0}) # mapping Loan_Status

# 2. Mandatory columns: drop rows where they are missing
print("Rows before:", len(df)) # check up
df = df.dropna(subset=["LoanAmount", "Loan_Amount_Term"]) # drops missing data of "LoanAmount", "Loan_Amount_Term" 
print("Rows after: ", len(df)) # check up

# Dependents: "3+" -> 3, treat as a number (it has a real order)
df["Dependents"] = df["Dependents"].replace("3+", 3).astype(float)

# defining X and y
X, y = df.drop(columns="Loan_Status"), df["Loan_Status"]

# putting each column on its category and because of credit_history needs to clasify them as objects
num_cols = ["ApplicantIncome", "CoapplicantIncome", "LoanAmount", "Loan_Amount_Term", "Dependents"]
cat_cols = ["Gender", "Married", "Education", "Self_Employed",
            "Credit_History", "Property_Area"]
X[cat_cols] = X[cat_cols].astype("object")

# 3. Preprocessing: fill missing values + StandardScaler on numeric features
pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])

# 4. Pipeline: normalization + SVC as one process
model = Pipeline([("pre", pre), ("svc", SVC())])

# 5. Split + train
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
model.fit(X_train, y_train)
print("Test accuracy: %.3f" % accuracy_score(y_test, model.predict(X_test)))

# 6. Save the trained model
joblib.dump(model, "loan_svc_model.pkl")
print("Model saved to loan_svc_model.pkl")

# 7. Collect all model data and save it to JSON
import json
from datetime import datetime
import sklearn
from sklearn.metrics import classification_report

svc = model.named_steps["svc"]
y_pred = model.predict(X_test)
scores = model.decision_function(X_test)   # distance from the decision boundary

model_info = {
    "file": {
        "model_file": "loan_svc_model.pkl",
        "data_file": "trainsvc.csv",
        "trained_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "sklearn_version": sklearn.__version__,
    },
    "features": {
        "numeric": num_cols,
        "categorical": cat_cols,
        "after_encoding": list(model.named_steps["pre"].get_feature_names_out()),
    },
    "samples": {
        "rows_total": len(X),
        "rows_train": len(X_train),
        "rows_test": len(X_test),
        "approved": int((y == 1).sum()),
        "rejected": int((y == 0).sum()),
        "examples": X.head(5).astype(object).where(X.head(5).notna(), None).to_dict(orient="records"),
    },
    "function": {
        "algorithm": "SVC",
        "kernel": svc.kernel,
        "C": svc.C,
        "gamma": svc.gamma,
        "intercept": float(svc.intercept_[0]),
        "support_vectors_total": int(svc.n_support_.sum()),
        "support_vectors_per_class": {"rejected": int(svc.n_support_[0]),
                                      "approved": int(svc.n_support_[1])},
    },
    "margin": {
        "explanation": "decision_function = signed distance from the boundary; "
                       "|value| < 1 means the sample is inside the margin",
        "min_score": round(float(scores.min()), 3),
        "max_score": round(float(scores.max()), 3),
        "mean_abs_score": round(float(abs(scores).mean()), 3),
        "test_samples_inside_margin": int((abs(scores) < 1).sum()),
    },
    "accuracy": {
        "test_accuracy": round(float(accuracy_score(y_test, y_pred)), 3),
        "report": classification_report(y_test, y_pred, output_dict=True,
                                         target_names=["Rejected", "Approved"]),
    },
}

with open("model_info.json", "w") as f:
    json.dump(model_info, f, indent=2, default=str)
print("Model info saved to model_info.json")


Rows before: 614
Rows after:  578
Test accuracy: 0.776
Model saved to loan_svc_model.pkl
Model info saved to model_info.json
